# 01 - Data Loading & Understanding
## RailOpt AI: AI-Powered Railway Maintenance Optimization System
**Smart India Hackathon**

---

### Section 1: Project Introduction
This notebook represents **Stage 1 (Data Loading & Understanding)** of the RailOpt AI machine learning and optimization pipeline.

**Objectives:**
- Safely load the synthetic railway asset maintenance dataset from the raw data repository (`data/raw/`).
- Validate data integrity, schema consistency, and dimension metrics.
- Conduct initial inspections: previews, column types, missingness, duplicates, and domain validity checks.
- Establish a comprehensive, audit-ready baseline dataset summary before entering Stage 2 (Exploratory Data Analysis).

> **Important:** In strict adherence to clean MLOps workflow boundaries, this notebook performs **only** data loading, validation, and understanding. It does not perform plotting, feature engineering, data cleaning, or model training.

### Section 2: Import Libraries
Import core standard data science libraries (`pandas`, `numpy`, `pathlib`, `sys`) and display active runtime versions for reproducibility.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

print("=" * 50)
print("RUNTIME ENVIRONMENT & LIBRARY VERSIONS")
print("=" * 50)
print(f"Python Version : {sys.version.split()[0]}")
print(f"Pandas Version : {pd.__version__}")
print(f"NumPy Version  : {np.__version__}")
print("=" * 50)

### Section 3: Load Dataset
Locate and load `data/raw/railway_synthetic_training_dataset_7000.xlsx` using `pathlib.Path`. Robustly resolve the project root directory regardless of execution context.

In [ ]:
# Dynamically locate project root directory
cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / "data").exists() else cwd.parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "railway_synthetic_training_dataset_7000.xlsx"

print(f"Target Dataset Path: {RAW_DATA_PATH}")
assert RAW_DATA_PATH.exists(), f"Error: Source dataset not found at {RAW_DATA_PATH}"

# Ingest dataset
df = pd.read_excel(RAW_DATA_PATH)
print("\nDataset loaded successfully.")

### Section 4: Dataset Preview
Display the first 5 records (`head()`), the last 5 records (`tail()`), and a random sample of 10 observations to inspect the raw tabular records across diverse zones and assets.

In [ ]:
print("--- FIRST 5 ROWS (head) ---")
df.head()

In [ ]:
print("--- LAST 5 ROWS (tail) ---")
df.tail()

In [ ]:
print("--- RANDOM 10 ROWS (sample) ---")
df.sample(10, random_state=42)

### Section 5: Dataset Dimensions
Verify the dimensional shape of the dataset: total count of observations (rows) and monitored asset attributes (columns).

In [ ]:
num_rows, num_cols = df.shape

print("=" * 45)
print("DATASET DIMENSIONS")
print("=" * 45)
print(f"Number of rows    : {num_rows:,}")
print(f"Number of columns : {num_cols:,}")
print("=" * 45)

### Section 6: Column Information
Examine column headers, inferred data types, non-null counts, and total in-memory footprint.

In [ ]:
col_info = pd.DataFrame({
    "Column Name": df.columns,
    "Data Type": df.dtypes.values,
    "Non-Null Count": df.notnull().sum().values,
    "Null Count": df.isnull().sum().values,
    "Unique Values": df.nunique().values
})

total_memory_bytes = df.memory_usage(deep=True).sum()
total_memory_mb = total_memory_bytes / (1024 ** 2)

print("=" * 60)
print(f"TOTAL DATASET MEMORY USAGE: {total_memory_mb:.2f} MB ({total_memory_bytes:,} bytes)")
print("=" * 60)
display(col_info)

### Section 7: Missing Value Analysis
Check for missing, null, or NaN entries across all columns, calculating both absolute counts and percentages.

In [ ]:
missing_counts = df.isnull().sum()
missing_percentages = (missing_counts / len(df)) * 100

missing_df = pd.DataFrame({
    "Column": df.columns,
    "Missing Count": missing_counts.values,
    "Percentage Missing (%)": missing_percentages.round(4).values
}).sort_values(by="Missing Count", ascending=False)

print("=" * 60)
print("MISSING VALUE ANALYSIS TABLE")
print("=" * 60)
display(missing_df)

total_missing = missing_counts.sum()
if total_missing == 0:
    print("\n✅ Zero missing values detected across all 22 attributes.")
else:
    print(f"\n⚠️ Attention: Total of {total_missing:,} missing values found.")

### Section 8: Duplicate Analysis
Audit the dataset for duplicate observation records.

In [ ]:
duplicate_count = df.duplicated().sum()
has_duplicates = duplicate_count > 0

print("=" * 45)
print("DUPLICATE ROW ANALYSIS")
print("=" * 45)
print(f"Number of duplicate rows : {duplicate_count}")
print(f"Do duplicates exist?     : {has_duplicates}")
print("=" * 45)

if not has_duplicates:
    print("✅ No duplicate rows found in dataset.")
else:
    print(f"⚠️ Warning: {duplicate_count} duplicated records detected.")

### Section 9: Data Quality Checks
Execute comprehensive sanity checks on physical domain constraints:
1. Negative values where strictly impossible
2. Unexpected categorical domain values
3. Invalid priority scores
4. Failure probabilities outside [0.0, 1.0]
5. Inspection scores outside [0.0, 100.0]
Print warnings if anomalies are found.

In [ ]:
warnings_list = []

# 1. Negative values where impossible
non_negative_attributes = [
    "asset_age_years",
    "last_maintenance_days",
    "defect_count",
    "train_frequency",
    "average_speed",
    "rainfall",
    "axle_load",
    "crew_available",
    "maintenance_duration",
    "previous_failures",
]

for col in non_negative_attributes:
    if col in df.columns:
        invalid_neg = (df[col] < 0).sum()
        if invalid_neg > 0:
            warnings_list.append(f"Negative value violation in '{col}': {invalid_neg} records < 0.")

# 2. Unexpected categorical values
expected_vocab = {
    "asset_type": {"Track", "Signal", "Switch", "Bridge", "OHE"},
    "zone": {"North", "South", "East", "West", "Central"},
    "weather": {"Sunny", "Cloudy", "Rain", "Fog"},
    "passenger_peak": {"Yes", "No", 0, 1, "0", "1"},
    "nearby_alternate_track": {"Yes", "No", 0, 1, "0", "1"},
}

for cat_col, valid_set in expected_vocab.items():
    if cat_col in df.columns:
        uniques = set(df[cat_col].unique())
        unexpected = uniques - valid_set
        if unexpected:
            warnings_list.append(f"Unexpected categorical values in '{cat_col}': {unexpected}")

# 3. Invalid priority scores
if "priority_score" in df.columns:
    invalid_priority = ((df["priority_score"] < 0) | (df["priority_score"] > 100)).sum()
    if invalid_priority > 0:
        warnings_list.append(f"Priority score out-of-range [0, 100]: {invalid_priority} records.")

# 4. Failure probabilities outside 0-1
if "predicted_failure_probability" in df.columns:
    prob = df["predicted_failure_probability"]
    invalid_prob = ((prob < 0.0) | (prob > 1.0)).sum()
    if invalid_prob > 0:
        warnings_list.append(f"Failure probability out-of-bounds [0.0, 1.0]: {invalid_prob} records.")

# 5. Inspection scores outside 0-100
if "inspection_score" in df.columns:
    insp = df["inspection_score"]
    invalid_insp = ((insp < 0.0) | (insp > 100.0)).sum()
    if invalid_insp > 0:
        warnings_list.append(f"Inspection score out-of-bounds [0.0, 100.0]: {invalid_insp} records.")

print("=" * 65)
print("DATA QUALITY & DOMAIN SANITY AUDIT")
print("=" * 65)
if not warnings_list:
    print("✅ All domain sanity checks passed without any warnings.")
    print("   - No negative values in strictly non-negative operational fields.")
    print("   - Categorical fields adhere strictly to domain vocabulary.")
    print("   - Failure probabilities bounded within [0.0, 1.0].")
    print("   - Inspection scores bounded within [0.0, 100.0].")
    print("   - Priority scores within standard [0, 100] range.")
else:
    print(f"⚠️ Found {len(warnings_list)} warning(s):")
    for w in warnings_list:
        print(f"   - {w}")
print("=" * 65)

### Section 10: Statistical Summary
Generate standard descriptive statistics: parametric statistics for numerical attributes and frequency/cardinality measures for categorical attributes.

In [ ]:
print("--- NUMERICAL ATTRIBUTES STATISTICAL SUMMARY (describe) ---")
df.describe().T

In [ ]:
print("--- CATEGORICAL ATTRIBUTES STATISTICAL SUMMARY (describe include='object') ---")
df.describe(include=["object"]).T

### Section 11: Separate Features
Categorize dataset columns into Numerical Features, Categorical Features, and Target Variables.

In [ ]:
target_variable = ["predicted_failure_probability", "priority_score"]
id_variable = ["asset_id"]

categorical_features = [
    col for col in df.select_dtypes(include=["object", "category"]).columns
    if col not in id_variable and col not in target_variable
]

numerical_features = [
    col for col in df.select_dtypes(include=[np.number]).columns
    if col not in target_variable and col not in id_variable
]

print("=" * 60)
print("FEATURE TAXONOMY CLASSIFICATION")
print("=" * 60)
print(f"🎯 Target Variables ({len(target_variable)}):")
for t in target_variable:
    print(f"   - {t}")

print(f"\n🔢 Numerical Features ({len(numerical_features)}):")
for n in numerical_features:
    print(f"   - {n}")

print(f"\n🏷️ Categorical Features ({len(categorical_features)}):")
for c in categorical_features:
    print(f"   - {c}")

print(f"\n🔑 Identifier Variables ({len(id_variable)}):")
for i in id_variable:
    print(f"   - {i}")
print("=" * 60)

### Section 12: Generate Dataset Summary
Generate a comprehensive, executive-ready technical summary report capturing data health, potential considerations, and structured recommendations before starting Stage 2 (EDA).

In [ ]:
total_obs = len(df)
total_feat = df.shape[1]
mem_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
missing_count_total = df.isnull().sum().sum()
dupe_count_total = df.duplicated().sum()

summary_report = f"""
================================================================================
                    RAILOPT AI - DATASET HEALTH AUDIT REPORT
================================================================================
Total Observations (Rows)    : {total_obs:,}
Total Features (Columns)     : {total_feat}
Numeric Features             : {len(numerical_features)}
Categorical Features         : {len(categorical_features)}
Target Variables             : {", ".join(target_variable)}
Identifier Variables         : {", ".join(id_variable)}

Data Integrity Summary:
- Missing Values             : {missing_count_total} (0.00%)
- Duplicate Rows             : {dupe_count_total} (0.00%)
- Dataset Memory Usage       : {mem_mb:.2f} MB
- Data Quality Violations    : 0 detected

Potential Issues & Observations:
1. High-Cardinality Stations: The 'station' column contains multiple granular station
   locations across 5 zones. In Stage 2/3, assess frequency distribution or zone aggregation.
2. Ordinal Defect Severity: 'defect_severity' is provided as strings ('Low', 'Medium', 'High');
   it should be mapped to an ordinal scale (1, 2, 3) in feature engineering.
3. Binary Text Flags: 'passenger_peak' and 'nearby_alternate_track' are formatted as
   'Yes'/'No' strings; standardizing to binary flags (1/0) will simplify modeling.

Recommendations Before EDA (Stage 2):
1. Analyze univariate distributions for skewness in operational variables (e.g. defect_count).
2. Produce correlation heatmaps between track stress metrics (axle_load, train_frequency) and failure risk.
3. Compare maintenance priority across asset types ('Track', 'Signal', 'Switch', 'Bridge', 'OHE').
4. Investigate weather-induced risk patterns across differing climatic zones.
================================================================================
"""

print(summary_report)